# 🎯 [K09_M09] 나만의 부대 생활 질문 목록 만들기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K09/k09_m09_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·상담 (Level 2) / K09 부대 정보 검색 시스템 |
| **미션** | 미션 09. 나만의 부대 생활 질문 목록 만들기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 프롬프트 엔지니어링 및 팩트체크 검증 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k09_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K09", "../uploads/K09"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k09_m01~m10) 준비 완료!")



---

## 🎯 미션 09. 나만의 부대 생활 질문 목록 만들기
**소요시간**: 약 1.5시간

![미션 9 10개 벤치마크 평가 질의 세트 및 정량 정확도 산출 다이어그램](/uploads/common/K09_m09_benchmark_eval.png)

### 📋 미션 개요 및 배경
완성 단계의 도우미 성능을 객관적으로 측정하기 위해 정상 질문 7개와 자료 밖 예외 질문 3개로 구성된 테스트 질의 세트(Benchmark)를 설계합니다. 사전에 정의된 모범 정답과 AI의 답변을 비교하여 정확도 지표를 정량적으로 산출하고, 오답 패턴을 분석해 최종 프롬프트 파라미터와 지시문을 튜닝합니다. AI 시스템의 품질을 스스로 측정하고 보완하는 테스트 주도 개선 역량을 기릅니다.

### 🎯 미션 학습목표
정상 질의 7개와 자료 밖 예외 질의 3개로 구성된 10개의 테스트 질문 세트를 설계하고, AI 답변의 정답률을 정량적으로 산출하여 미흡한 프롬프트를 반복 교정합니다. **이를 통해 AI 시스템의 성능을 정량 평가하는 벤치마크 테스트 방법론과 오류 분석 기반의 반복 개선(Iterative Refinement) 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 시험 질문 10개(정상 7개, 자료 밖 3개)를 선정한다.
#### 이 단계의 역할
도우미의 종합 성능을 체계적으로 측정할 수 있는 10개의 벤치마크 테스트 질문 세트(자료 내 정상 질의 7건, 자료 밖 예외 질의 3건)를 구성합니다.

#### 해야 할 일

<li>정상 질문 7건(시간, 위치, 문의처, 예외 조건 등 다양한 카테고리 골고루 분배)을 설계합니다.

</li><li>자료 밖 질문 3건(포상휴가 기준, 두발 규정, 사격 훈련 일정 등 부대 내 실제 자주 묻지만 자료에는 없는 항목)을 구성합니다.

</li><li>AI를 활용하여 질문의 모호성을 검토하고 10개 질문의 대표성을 확보합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[벤치마크 테스트 질문 세트 설계 질의]:** "부대 안내 도우미 성능을 평가하기 위해, 제공된 자료표에서 답변할 수 있는 다양한 유형의 정상 질문 7개와, 자료에는 없지만 신병들이 실제로 자주 묻는 자료 밖 질문 3개를 만들어 줘.
[자료표]: {갱신된 자료표}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **질문 비율 준수:** 정상 질문 7건과 자료 밖 질문 3건의 7:3 비율이 정확히 준수되었는가?
- **질문 다양성:** 단순 위치 질문 외에 소독 시간이나 예약제 등 조건부 질문이 포함되었는가?

#### 핵심 확인 지표
<blockquote>10개 벤치마크 시험 질문 세트

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>시험 질문 10개 목록

</li>

#### 🔹 Step 2. 시험 질문 10개에 대한 모범 정답표를 작성한다.
#### 이 단계의 역할
AI 답변의 채점 기준이 될 10개 질문의 정답 기준(Ground Truth)을 사전에 명확히 작성합니다.

#### 해야 할 일

<li>정상 질문 7건에 대해 자료표 원문과 100% 일치하는 모범 정답(시간, 위치, 근거)을 기술합니다.

</li><li>자료 밖 질문 3건에 대해 '자료 미등록 안내 및 행정반/당직실 문의' 표준 거절 응답을 정답으로 명시합니다.

</li><li>채점관 관점에서 핵심 키워드 채점 기준(채점 루브릭)을 정리합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[모범 정답표(Ground Truth) 작성 질의]:** "위 10개 시험 질문 각각에 대해, 안내 자료표와 예외 처리 원칙에 근거한 무결점 모범 정답(Ground Truth)과 필수 포함 핵심 키워드를 표 형태로 작성해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **정답 무결성:** 정상 질문 정답이 최신 자료표와 한 글자도 어긋남이 없는가?
- **예외 정답 일관성:** 자료 밖 질문 3건의 정답이 모두 표준 Fallback 문구로 지정되었는가?

#### 핵심 확인 지표
<blockquote>10개 시험 질문 모범 정답표 및 채점 루브릭

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>시험 질문 10개와 정답(필수 산출물)

</li>

#### 🔹 Step 3. 도우미 답과 모범 정답을 비교해 정확도를 계산한다.
#### 이 단계의 역할
현재 구축된 도우미 프롬프트에 10개 시험 질문을 일괄 입력하고, 모범 정답과 비교하여 정량적 정확도(Score %)를 산출합니다.

#### 해야 할 일

<li>완성된 프롬프트에 10개 질문을 순차적으로 입력하고 응답을 기록합니다.

</li><li>각 문항별로 정답(1점), 부분정답(0.5점), 오답(0점)으로 채점합니다.

</li><li>총점과 정확도 백분율(예: 8.5/10 = 85%)을 계산하고 오답 문항을 특정합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[벤치마크 자동 채점 및 정확도 산출 질의]:** "아래 [AI 응답 10건]을 앞서 작성한 [모범 정답 10건]과 1:1로 채점해 줘. 각 문항별 일치 여부(PASS/FAIL)와 감점 사유, 최종 정확도(%)를 계산해 줘.
[모범 정답]: {Step 2 정답표}
[AI 응답]: {수집된 응답}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **채점 객관성:** 서식 누락, 시간 오차, 불완전 거절 등을 엄격히 감점 처리했는가?
- **정확도 산출:** 10개 문항에 대한 정량 지표(%)가 수학적으로 정확히 계산되었는가?

#### 핵심 확인 지표
<blockquote>10개 문항 채점 결과표 및 정확도(%) 산출 기록

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>정확도 기록(필수 산출물)

</li>

#### 🔹 Step 4. 틀린 질문을 분석하여 프롬프트를 개선한다.
#### 이 단계의 역할
감점이나 오답이 발생한 문항의 취약점을 분석하고, 이를 극복할 수 있도록 프롬프트 지시문과 제약조건을 튜닝합니다.

#### 해야 할 일

<li>오답이 발생한 문항(조건부 시간 누락, 질문의 의도 오독 등)의 근본 원인을 파악합니다.

</li><li>프롬프트 내 지시사항 보강, 예시(Few-shot) 추가, 금지 조항 구체화 등 수정 작업을 수행합니다.

</li><li>개선된 프롬프트로 틀렸던 질문을 재시험하여 정답(정확도 향상)으로 개선되었는지 재검증합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[오답 분석 기반 프롬프트 튜닝 질의]:** "질문 4번에서 AI가 점심 소독 시간 안내를 누락하여 감점되었어. 프롬프트의 지시사항 중 어떤 문구를 수정해야 향후 소독 시간 같은 세부 조건을 절대 누락하지 않고 완벽히 출력할 수 있을지 개선안을 제시해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **재시험 통과:** 프롬프트 수정 후 취약 문항이 완벽하게 정답으로 개선되었는가?
- **회귀 오류 방지:** 수정 과정에서 다른 정상 문항의 답변 품질이 저하되지 않았는가?

#### 핵심 확인 지표
<blockquote>취약점 개선 전후 비교표 및 최종 튜닝 프롬프트

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>수정 프롬프트(필수 산출물)

</li>

#### 🔹 Step 5. 심화 도전 과제 및 최적화
#### 이 단계의 역할
동료 장병(또는 가상 짝)의 도우미 프롬프트와 내 시험 질문 세트를 상호 교환(Cross-testing)하여 시스템의 일반화 성능을 검증합니다.

#### 해야 할 일
짝의 도우미에 내 시험 질문을 넣어 비교해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[교차 검증 질의]:** "다른 사용자가 만든 다음 5개의 돌발 질문을 내 도우미 프롬프트에 넣었을 때의 견고성(Robustness)을 평가해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **미지의 질문 방어:** 다른 사람의 독특한 질문 스타일에도 5단 서식과 정확도가 유지되는가?

#### 핵심 확인 지표
<blockquote>상호 교차 테스트 평가 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 교차 검증 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M09] 나만의 부대 생활 질문 목록 만들기 실습 가이드 및 작성 서식

## 1. 10개 벤치마크 평가 질의 및 모범 정답표 (Ground Truth)

| 번호 | 유형 | 시험 질문 내용 | 모범 정답 요약 (Ground Truth) | 필수 포함 키워드 |
| :---: | :---: | :--- | :--- | :--- |
| **01** | 정상 | 의무실 점심 소독 시간에 진료 가능한가? | 12:00~13:30 소독으로 일반 진료 불가 | 12:00~13:30, 소독, 진료 불가 |
| **02** | 정상 | 체단실 평일 마감 시간은 몇 시인가? | 평일 20:00 마감 (단축 운영 반영) | 평일 20:00 |
| **03** | 정상 | 충성마트(PX) 일요일 이용 가능한가? | 일요일 정기 휴무로 이용 불가 | 일요일, 휴무 |
| **04** | 정상 | 병영도서관 대출 권수와 기한은? | 1인 3권, 14일간 대출 가능 | 3권, 14일 |
| **05** | 정상 | 이발실 바로 가면 이용 가능한가? | 화·목·토 사전 예약제, 즉시 방문 불가 | 사전 예약제, 선착순 불가 |
| **06** | 정상 | 사제 택배 당일 수령 가능한가? | 보안검색 후 익일 17:30~18:30 수령 | 익일, 17:30~18:30 |
| **07** | 정상 | 토요일 아침 식사 시간은 언제인가? | 브런치데이 운영으로 09:00~10:30 | 브런치, 09:00~10:30 |
| **08** | 예외 | 정기휴가 심사 발표일은 언제인가? | 안내 자료 미등록 정보 (행정반 문의) | 자료 미등록, 행정반 문의 |
| **09** | 예외 | 외박 신청에 필요한 상점 점수는? | 안내 자료 미등록 정보 (행정반 문의) | 자료 미등록, 행정반 문의 |
| **10** | 예외 | 다음 달 유격 훈련 장소는 어디인가? | 안내 자료 미등록 정보 (행정반 문의) | 자료 미등록, 행정반 문의 |

---

## 2. 벤치마크 채점표 및 최종 정확도 기록
- **정상 질문 정답률:** 7 / 7 (100%)
- **자료 밖 질문 차단율:** 3 / 3 (100%)
- **서식(5단 템플릿) 준수율:** 10 / 10 (100%)
- **종합 정확도 (Score):** **100% (10 / 10 PASS)**

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **10대 벤치마크 평가 질의 세트 양식 (정상 7건 : 예외 3건):** 대표성 있는 질의 문항 설계 시트
- **모범 정답표(Ground Truth) 및 채점 루브릭 (Scoring Rubric):** 정답, 부분정답, 오답 채점 기준 및 핵심 키워드 체크리스트
- **[학습 가이드] LLM 정량 평가(Evaluation) 방법론 및 반복 프롬프트 튜닝:** 정답률(Accuracy %) 산출과 오류 패턴 분석을 통한 프롬프트 개선 사이클

**실습 팁:** 10개 질문 중 정상 질문 7개는 시간, 장소, 절차 등 시설을 고루 섞고, 예외 질문 3개는 신병들이 진짜 궁금해할 법한 규정 외 질문(월급, 훈련 등)으로 구성하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 나만의 활용법(시험·개선) — 정상 질의와 자료 밖 질의로 10개 벤치마크를 구축하고 정답률 산출 및 오답 원인을 프롬프트에 개선했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>10개 벤치마크 평가 질의 및 모범 정답표(Ground Truth) 작성 완료

</li><li>10개 문항 채점 결과표 및 정량 정확도(%) 산출 기록

</li><li>취약점 보완을 위한 프롬프트 수정 전후 비교표 및 최종 튜닝 프롬프트 도출

</li>

#### 채점자 판정 메모
<blockquote>정답률 수치 자체보다, 채점 결과 틀린 문항의 원인을 정확히 짚어내고 프롬프트를 고쳐서 개선해 냈는지를 중점 평가한다.

</blockquote>





In [ ]:
# [미션 09. 나만의 부대 생활 질문 목록 만들기] 학생 실습 프롬프트 및 결과 기록란
mission_09_prompt = """
# [TODO: 미션 09] 정상 질문 7건과 자료 밖 예외 질문 3건으로 구성된 10대 벤치마크 세트를 설계하고 정확도를 평가하는 프롬프트를 작성하세요.
# 힌트: load_dataset(9)의 benchmarkSet을 확인하고, Ground Truth 대조 채점표 및 최종 정답률 산출 방식을 구성합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_09_result = """
### 📋 [미션 09 실습 결과]
## 1. 10개 벤치마크 채점표
| 번호 | 유형 | 시험 질문 내용 | 도우미 답변 평가 | 채점 (PASS/FAIL) |
| :---: | :---: | :--- | :--- | :---: |
| **01** | 정상 | 의무실 소독 시간 진료 여부 | | |
| **02** | 정상 | 사지방 화요일 점검 시간 | | |
| **03** | 정상 | PX 토요일 운영 시간 | | |
| **04** | 정상 | 체단실 수요일 방역 시간 | | |
| **05** | 정상 | 도서관 대출 권수 및 반납 기한 | | |
| **06** | 정상 | 세탁기 고장 신고처 | | |
| **07** | 정상 | 이발실 휴무일 및 예약 | | |
| **08** | 예외 | 당직 사령 근무 수당 | | |
| **09** | 예외 | 주말 외출/외박 가능 시간 | | |
| **10** | 예외 | 영내 노래방 코인 가격 | | |

- **최종 정확도 산출**: (   ) / 10 = (   )%
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 09. 나만의 부대 생활 질문 목록 만들기", mission_09_prompt, mission_09_result)

